# PrintWatch - Entrenamiento del modelo de detección de defectos

Bakeoff YOLO11s vs YOLOv8n, entrenamiento completo del ganador, validación y export a ONNX,
para reemplazar `inference/models/yolo11s_dummy.onnx`.

**Antes de correr esto**: subí la carpeta `inference/training/` de tu repo a Google Drive
(por ejemplo a `MyDrive/DeteccionImpresion3d/inference/training/`). Drive se usa solo para
**leer** `scripts/` y `configs/`, y para **guardar** el resultado final que tiene que persistir
entre sesiones de Colab (que se resetean): el dataset ya procesado (empaquetado en un solo
`.tar`) y el modelo entrenado.

**Todo el I/O pesado (descarga, conversión, split, entrenamiento) corre en el disco local de
Colab (`/content/local_work/`), no en Drive**: Drive tiene latencia de red por operación de
archivo, y con ~11.000 archivos chicos entre imágenes y labels, copiarlos uno por uno tarda
más de una hora en vez de minutos. Consecuencia: si la sesión de Colab se corta a mitad de
camino, lo que no se haya persistido a Drive todavía se pierde (ver las celdas marcadas como
"Persistir" más abajo).

Ver `README.md` en esa misma carpeta para los pasos manuales completos.

Clases (orden fijo, debe coincidir con `datasets/dataset.yaml` y `configs/train_config.yaml`):
`spaghetti, warping, stringing, layer_shift, under_extrusion, blob_over_extrusion`.


## 1. Setup: Google Drive + dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics roboflow pyyaml

Ajustá `DRIVE_PROJECT_DIR` si subiste la carpeta a otro lugar de tu Drive. El resto de las rutas
(`LOCAL_WORK_DIR` y lo que cuelga de ahí) son disco local de Colab y no hace falta tocarlas.


In [ ]:
from pathlib import Path

# --- Drive: SOLO entradas livianas (scripts/configs) y resultados finales a persistir. ---
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/DeteccionImpresion3d/inference/training')
assert DRIVE_PROJECT_DIR.is_dir(), (
    f'No se encontró {DRIVE_PROJECT_DIR}. Subí la carpeta inference/training/ de tu repo a Drive primero '
    '(ver README.md de esa carpeta).'
)
SCRIPTS_DIR = DRIVE_PROJECT_DIR / 'scripts'
CONFIGS_DIR = DRIVE_PROJECT_DIR / 'configs'
DRIVE_PERSIST_DIR = DRIVE_PROJECT_DIR / 'persisted'  # acá caen dataset procesado (.tar) y modelo final

# --- Disco local de Colab: todo el I/O pesado (miles de archivos chicos) va acá. ---
LOCAL_WORK_DIR = Path('/content/local_work')
DATASETS_DIR = LOCAL_WORK_DIR / 'datasets'
RAW_DIR = DATASETS_DIR / 'raw'
INTERIM_DIR = DATASETS_DIR / 'interim'
PROCESSED_DIR = DATASETS_DIR / 'processed'
RUNS_DIR = LOCAL_WORK_DIR / 'runs'

for d in (RAW_DIR, INTERIM_DIR, PROCESSED_DIR, RUNS_DIR, DRIVE_PERSIST_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('Drive (entradas/resultados):', DRIVE_PROJECT_DIR)
print('Disco local (trabajo pesado):', LOCAL_WORK_DIR)


## 2. Cargar hiperparámetros desde `configs/train_config.yaml`

Todo lo parametrizable vive en ese archivo, no acá - si querés cambiar épocas, batch size,
el split, etc., editá el YAML (en Drive o en tu repo y volvé a subirlo) y volvé a correr esta celda.


In [ ]:
import yaml

with open(CONFIGS_DIR / 'train_config.yaml') as f:
    CONFIG = yaml.safe_load(f)

CLASS_NAMES = CONFIG['classes']
print(yaml.dump(CONFIG, sort_keys=False))

### 2b. (Solo sesión nueva) Restaurar el dataset ya procesado desde Drive

Si ya corriste la conversión/split en una sesión anterior y quedó guardado el
`persisted/processed.tar` en Drive (ver la celda "Persistir" en la sección 4), **corré esta
celda y saltá directo a la celda que genera `dataset.runtime.yaml`** (al final de la sección 4),
salteándote la descarga, la conversión y el split: restaurar es copiar **un solo archivo** de
Drive y descomprimirlo en el disco local, cuestión de minutos.

Si es la primera vez (no hay `.tar` todavía), esta celda solo avisa que no encontró nada -
seguí de largo con la sección 3 normalmente.


In [ ]:
import shutil

drive_archive = DRIVE_PERSIST_DIR / 'processed.tar'

if drive_archive.is_file():
    local_archive = LOCAL_WORK_DIR / 'processed.tar'
    shutil.copy2(drive_archive, local_archive)  # 1 archivo grande: rápido, a diferencia de miles chicos
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(local_archive), extract_dir=str(PROCESSED_DIR))
    local_archive.unlink()

    n_images = sum(1 for p in (PROCESSED_DIR / 'images').rglob('*') if p.is_file())
    print(f'Restaurado desde {drive_archive}: {n_images} imágenes en {PROCESSED_DIR}')
    print('-> Saltá a la celda que genera dataset.runtime.yaml (final de la sección 4).')
else:
    print(f'No hay {drive_archive} todavía - seguí con la sección 3 (descarga) normalmente.')


## 3. Descargar datasets desde Roboflow Universe

Necesitás una cuenta gratuita de Roboflow y tu API key (Settings -> API Keys en
app.roboflow.com). Ya hay 3 datasets reales confirmados cargados abajo, cubriendo las 6 clases
entre los tres (ver la tabla de cobertura en el README de esta carpeta). Si más adelante sumás
más, agregá otro bloque igual a los de abajo (con otro `dataset_name`) y otra entrada en
`datasets_to_convert` en la sección de conversión.

Los datasets se descargan a `RAW_DIR`, que es **disco local de Colab** (no Drive).


In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass('Roboflow API key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [ ]:
# Dataset 1: "3D PRINT DEFECT DETECTION" por RVM Detection - 3612 imágenes, 6 clases
# (confirmadas desde el data.yaml real del dataset, en inglés pese a que la interfaz de
# Roboflow las mostraba traducidas): blobs (1119), crack (713), Layer_shifting (629),
# spaghetti (2042), stringing (1924), Warping (805). Este dataset solo ya cubre 5 de
# nuestras 6 clases con buen volumen - falta under_extrusion.
dataset_name = 'rvm_defect_detection'
project = rf.workspace('samuel-rodriguez-2vtvz').project('3d-print-defect-detection-6fcmi-drylp')
version = project.version(1)
dataset = version.download('yolov8', location=str(RAW_DIR / dataset_name))
print('Descargado en:', dataset.location)


In [ ]:
# Dataset 2: "3d print fail detection" - 1002 imágenes, 4 clases (3D-print-fail-detection
# con 0 anotaciones - se ignora sola -, crack, spaghetti [108], stringing [672]).
dataset_name = 'print_fail_detection'
project = rf.workspace('samuel-rodriguez-2vtvz').project('3d-print-fail-detection-jckut')
version = project.version(1)
dataset = version.download('yolov8', location=str(RAW_DIR / dataset_name))
print('Descargado en:', dataset.location)


In [ ]:
# Dataset 3: "Under Extrusion" - 999 imágenes, 1 clase (Under Extrusion, 1414
# anotaciones). Completa la única clase que los datasets 1 y 2 no cubrían.
dataset_name = 'under_extrusion_ds'
project = rf.workspace('samuel-rodriguez-2vtvz').project('under-extrusion-pf0uy-nlyil')
version = project.version(1)
dataset = version.download('yolov8', location=str(RAW_DIR / dataset_name))
print('Descargado en:', dataset.location)


**The Spaghetti Detective**: no encontré un dataset público descargable directamente para
detección de objetos (es un servicio de monitoreo con su propio modelo privado de detección de
fallos, no un dataset abierto con descarga formal). Si en el futuro publican uno, el lugar para
sumarlo es acá, con el mismo patrón: bajarlo a `RAW_DIR / 'spaghetti_detective'` y convertirlo
con `convert_to_yolo.py` como los de Roboflow. Por ahora queda como placeholder.

## 4. Convertir a nuestra taxonomía de 6 clases y generar el split

`convert_to_yolo.py` remapea las clases de cada dataset fuente a nuestras 6 clases. El mapeo
de los 3 datasets de la sección anterior ya está confirmado y cargado en
`MAPPINGS['roboflow_generic']` (en `scripts/convert_to_yolo.py`): `blobs`->blob_over_extrusion,
`layer_shifting`->layer_shift, `spaghetti`->spaghetti, `stringing`->stringing,
`warping`->warping, `under extrusion`->under_extrusion. `crack` (datasets 1 y 2) y
`3D-print-fail-detection` (clase auto-generada por Roboflow, 0 anotaciones) quedan sin mapear
a propósito - no corresponden a ninguna de nuestras 6 clases.

Entre los 3 datasets ya hay volumen real para las 6 clases - ver la tabla de cobertura en el
`README.md` de esta carpeta.

`merge_datasets.py` junta todo y arma el split train/val/test **estratificado por secuencia**,
no por imagen individual, para no filtrar frames casi duplicados entre splits.

Conversión y split corren enteros en disco local (`INTERIM_DIR` / `PROCESSED_DIR`) - minutos,
no horas. Recién al final de la sección se persiste el resultado a Drive, empaquetado.


In [ ]:
import subprocess

# Uno por cada dataset bajado en la sección anterior - agregá una línea más acá si sumás
# otro dataset a futuro (con el mismo dataset_name que le pusiste al bajarlo).
datasets_to_convert = [
    ('rvm_defect_detection', 'roboflow_generic'),
    ('print_fail_detection', 'roboflow_generic'),
    ('under_extrusion_ds', 'roboflow_generic'),
]

for name, mapping in datasets_to_convert:
    subprocess.run([
        'python', str(SCRIPTS_DIR / 'convert_to_yolo.py'),
        '--source', str(RAW_DIR / name),
        '--dataset-name', name,
        '--mapping', mapping,
        '--output', str(INTERIM_DIR),
    ], check=True)


In [ ]:
split = CONFIG['dataset']['train_val_test_split']
seed = CONFIG['dataset']['seed']

subprocess.run([
    'python', str(SCRIPTS_DIR / 'merge_datasets.py'),
    '--interim', str(INTERIM_DIR),
    '--output', str(PROCESSED_DIR),
    '--train', str(split[0]), '--val', str(split[1]), '--test', str(split[2]),
    '--seed', str(seed),
], check=True)

**Persistir a Drive**: empaqueta `processed/` en **un solo `.tar`** (local, segundos) y copia
ese único archivo a Drive. Copiar `processed/` a Drive archivo por archivo reproduciría
exactamente el cuello de botella que estamos evitando; un solo archivo grande, en cambio, sube
rápido. En una sesión nueva, la celda 2b lo restaura.


In [ ]:
import shutil
import time

t0 = time.time()
local_archive = Path(shutil.make_archive(str(LOCAL_WORK_DIR / 'processed'), 'tar', root_dir=PROCESSED_DIR))
print(f'Empaquetado local: {local_archive.stat().st_size / 1e6:.0f} MB en {time.time() - t0:.0f}s')

drive_archive = DRIVE_PERSIST_DIR / 'processed.tar'
t0 = time.time()
shutil.copy2(local_archive, drive_archive)
assert drive_archive.stat().st_size == local_archive.stat().st_size, 'La copia a Drive quedó incompleta'
print(f'Persistido en Drive: {drive_archive} ({time.time() - t0:.0f}s)')
local_archive.unlink()


In [ ]:
# dataset.yaml apunta a processed/ con una ruta relativa (para poder commitearlo al repo) -
# acá reescribimos una copia con la ruta absoluta LOCAL de processed/, que es la que YOLO usa.
# (Se lee desde Drive - es un archivo chico - y la copia runtime se escribe en disco local.)
with open(DRIVE_PROJECT_DIR / 'datasets' / 'dataset.yaml') as f:
    dataset_yaml = yaml.safe_load(f)
dataset_yaml['path'] = str(PROCESSED_DIR)

RUNTIME_DATASET_YAML = DATASETS_DIR / 'dataset.runtime.yaml'
with open(RUNTIME_DATASET_YAML, 'w') as f:
    yaml.dump(dataset_yaml, f, sort_keys=False)

print(RUNTIME_DATASET_YAML.read_text())


## 5. Bakeoff: YOLO11s vs YOLOv8n

Pocas épocas cada uno (parametrizable en `train_config.yaml` -> `bakeoff`), comparando mAP50
por clase - con foco especial en **stringing**, la clase más difícil (objetos finos, bajo
contraste), no solo en el mAP50 promedio.

In [ ]:
from ultralytics import YOLO

bakeoff_cfg = CONFIG['bakeoff']
priority_class = bakeoff_cfg['priority_class']
priority_class_idx = CLASS_NAMES.index(priority_class)

bakeoff_results = {}

for model_name in bakeoff_cfg['models']:
    print(f'\n=== Bakeoff: {model_name} ===')
    model = YOLO(model_name)
    model.train(
        data=str(RUNTIME_DATASET_YAML),
        epochs=bakeoff_cfg['epochs'],
        imgsz=bakeoff_cfg['imgsz'],
        batch=bakeoff_cfg['batch'],
        patience=bakeoff_cfg['patience'],
        project=str(RUNS_DIR / 'bakeoff'),
        name=model_name.replace('.pt', ''),
        exist_ok=True,
    )
    metrics = model.val(data=str(RUNTIME_DATASET_YAML))
    per_class_map50 = metrics.box.ap50  # array indexed by class
    bakeoff_results[model_name] = {
        'overall_map50': float(metrics.box.map50),
        'stringing_map50': float(per_class_map50[priority_class_idx]),
        'per_class_map50': {CLASS_NAMES[i]: float(v) for i, v in enumerate(per_class_map50)},
    }

In [ ]:
print(f"{'model':<15}{'overall mAP50':<18}{priority_class + ' mAP50':<20}")
for model_name, r in bakeoff_results.items():
    print(f"{model_name:<15}{r['overall_map50']:<18.4f}{r['stringing_map50']:<20.4f}")

# Ganador por mAP50 de la clase prioritaria (stringing), no por el mAP50 general -
# es la clase que más nos importa mejorar, per configs/train_config.yaml.
winner = max(bakeoff_results, key=lambda m: bakeoff_results[m]['stringing_map50'])
print(f"\nGanador del bakeoff: {winner}")

## 6. Entrenamiento completo del modelo ganador

Usa los hiperparámetros de `train_config.yaml` -> `final_training`. Si querés forzar un modelo
distinto al que ganó el bakeoff, cambiá `CONFIG['final_training']['model']` antes de correr esto.

In [ ]:
import shutil

final_cfg = CONFIG['final_training']
final_model_name = final_cfg.get('model') or winner
print('Entrenando modelo final:', final_model_name)

final_model = YOLO(final_model_name)
final_model.train(
    data=str(RUNTIME_DATASET_YAML),
    epochs=final_cfg['epochs'],
    imgsz=final_cfg['imgsz'],
    batch=final_cfg['batch'],
    patience=final_cfg['patience'],
    optimizer=final_cfg['optimizer'],
    lr0=final_cfg['lr0'],
    seed=final_cfg['seed'],
    project=str(RUNS_DIR / 'final'),
    name=final_model_name.replace('.pt', ''),
    exist_ok=True,
)
# Un solo archivo chico: persistir los pesos entrenados a Drive por si la sesión se corta
# antes del export (no cuesta nada y evita repetir horas de entrenamiento).
best_pt = RUNS_DIR / 'final' / final_model_name.replace('.pt', '') / 'weights' / 'best.pt'
if best_pt.is_file():
    shutil.copy2(best_pt, DRIVE_PERSIST_DIR / 'best.pt')
    print('Pesos persistidos en Drive:', DRIVE_PERSIST_DIR / 'best.pt')
else:
    print('No encontré', best_pt, '- revisá RUNS_DIR/final/ a mano.')


## 7. Validación: mAP50 / mAP50-95 por clase + matriz de confusión

In [ ]:
val_metrics = final_model.val(data=str(RUNTIME_DATASET_YAML), split='test')

print(f"{'class':<20}{'mAP50':<10}{'mAP50-95':<10}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<20}{val_metrics.box.ap50[i]:<10.4f}{val_metrics.box.ap[i]:<10.4f}")
print(f"\n{'overall':<20}{val_metrics.box.map50:<10.4f}{val_metrics.box.map:<10.4f}")

In [ ]:
from IPython.display import Image as IPImage, display

# Ultralytics guarda esto automáticamente en la carpeta del run de validación.
confusion_matrix_path = Path(val_metrics.save_dir) / 'confusion_matrix_normalized.png'
if confusion_matrix_path.is_file():
    display(IPImage(filename=str(confusion_matrix_path)))
else:
    print('No se encontró la matriz de confusión en', confusion_matrix_path)

## 8. Exportar a ONNX

In [ ]:
import hashlib
import shutil

export_cfg = CONFIG['export']
onnx_path = final_model.export(
    format=export_cfg['format'],
    simplify=export_cfg['simplify'],
    opset=export_cfg['opset'],
    imgsz=export_cfg['imgsz'],
)
print('Exportado a:', onnx_path)

# IMPORTANTE: copiar el FP32 YA. Al exportar INT8 (celda 8b) Ultralytics genera <nombre>_int8.onnx y BORRA el
# .onnx FP32 de origen (en 8.4.x: `source.unlink()`), así que `onnx_path` deja de existir después de 8b.
ONNX_DIR = LOCAL_WORK_DIR / 'onnx'
ONNX_DIR.mkdir(parents=True, exist_ok=True)
onnx_fp32_path = ONNX_DIR / 'best_fp32.onnx'
shutil.copy2(onnx_path, onnx_fp32_path)
# Guarda de cordura: un FP32 no tiene nodos de cuantización. (Ya nos pasó tener un "FP32" que era el INT8.)
assert b'QuantizeLinear' not in onnx_fp32_path.read_bytes(), 'Este .onnx está cuantizado - no es FP32.'
shutil.copy2(onnx_fp32_path, DRIVE_PERSIST_DIR / 'best_fp32.onnx')
print('FP32 guardado en', onnx_fp32_path, 'y en Drive:', DRIVE_PERSIST_DIR / 'best_fp32.onnx')

### 8b. (Opcional) Cuantización INT8

Reduce tamaño/latencia a costa de algo de precisión - necesita pasar el dataset como
calibración. Correr esta celda solo si querés probar la versión cuantizada.

**Cuidado**: Ultralytics genera `best_int8.onnx` y **borra el `best.onnx` FP32** de origen. El FP32 ya
quedó copiado en la celda anterior (`onnx_fp32_path` y `persisted/best_fp32.onnx`); usá esas copias, no
`onnx_path`. Además, en 8.4.x `int8=True` es un alias de `quantize=8` (INT8 estático QDQ con
calibración).

In [ ]:
onnx_int8_path = final_model.export(
    format=export_cfg['format'],
    simplify=export_cfg['simplify'],
    opset=export_cfg['opset'],
    imgsz=export_cfg['imgsz'],
    int8=True,
    data=str(RUNTIME_DATASET_YAML),
)
print('Exportado (INT8) a:', onnx_int8_path)
assert b'QuantizeLinear' in Path(onnx_int8_path).read_bytes(), 'Este .onnx NO está cuantizado.'
shutil.copy2(onnx_int8_path, DRIVE_PERSIST_DIR / 'best_int8.onnx')
print('INT8 guardado en Drive:', DRIVE_PERSIST_DIR / 'best_int8.onnx')

### 8c. Calibrar el umbral de confianza (precisión/recall por clase vs. umbral)

Mide, sobre el **test set completo**, precisión y recall de cada clase para umbrales de 0.05 a 0.70,
y sugiere umbrales para `CONFIDENCE_THRESHOLD` en `inference/config.py` (o por clase, si las
curvas lo justifican). Reemplaza el "0.25 por defecto" y las conclusiones sacadas de unas pocas fotos.

Puntos importantes:
- Evalúa **el `.onnx` que se despliega** (el INT8 de 8b, o `persisted/best_int8.onnx` en una sesión nueva) con
  el **mismo pre/postprocesado que el worker** (resize directo a 640x640, NMS agnóstico a la clase, IoU 0.45), no
  con `val()` de Ultralytics (que usa letterbox y NMS por clase). Por eso los números pueden diferir un poco de
  la sección 7; la comparación de la última tabla es solo una verificación de orden de magnitud.
- `JPEG_REENCODE = True` re-codifica cada imagen como JPEG q95 antes de inferir, igual que `capture` en
  producción: los scores del INT8 son muy escalonados y una detección borderline puede caer por debajo del umbral
  solo por eso (en fotos reales: 0.236 -> 0.172).
- Además de precisión/recall por **instancia** (IoU >= 0.5) reporta el nivel **imagen** ("¿se marcó esta clase en el
  frame?"), que es el que usa el sistema (`has_defect`, alerta de atasco de 3 capas), y cuántos frames sin ninguna
  anotación se marcan igualmente (falsas alarmas).
- Clases con pocas instancias en el test (`blob_over_extrusion`, probablemente) dan curvas ruidosas: la tabla
  lo avisa. No sobreajustes un umbral a 30 instancias.
- **Etiquetas de segmentación**: igual que Ultralytics, un archivo de etiquetas solo de polígonos se convierte a la
  caja que los encierra (no se pierde el ground truth), y un archivo que **mezcla** polígonos y cajas, o está malformado,
  excluye esa imagen entera con un warning (`ignoring corrupt image/label`). Al final se listan las excluidas (también
  en el JSON). Si se excluye más del 5% de las imágenes la celda aborta: eso ya no son "unas pocas corruptas".
- Tarda unos minutos (una inferencia por imagen en CPU); se guarda todo en Drive (`persisted/calibration/`).

In [ ]:
!pip install -q onnxruntime

import shutil
import sys

# --- Modelo a calibrar: el INT8 recién exportado en 8b, o el que hayas dejado en persisted/. ---
# Para el FP32 (celda 8): CALIBRATION_MODEL = onnx_fp32_path  o  DRIVE_PERSIST_DIR / 'best_fp32.onnx'
CALIBRATION_MODEL = Path(onnx_int8_path) if 'onnx_int8_path' in globals() else DRIVE_PERSIST_DIR / 'best_int8.onnx'
assert CALIBRATION_MODEL.is_file(), (
    f'No encontré {CALIBRATION_MODEL}. Corré la celda 8b (INT8) o copiá best_int8.onnx a {DRIVE_PERSIST_DIR}.'
)
JPEG_REENCODE = True   # como capture en producción; False = imágenes tal cual están en el test set
LIMIT = None           # p. ej. 50 para una prueba rápida; None = las 841 imágenes completas
# Opcional: tus métricas de Ultralytics para la verificación de orden de magnitud (mismo archivo que usa el worker).
REFERENCE_METRICS = None   # p. ej. DRIVE_PROJECT_DIR.parent.parent / 'models' / 'best_int8.metrics.json'

sys.path.insert(0, str(SCRIPTS_DIR))
import calibrate_thresholds

# Una carpeta por modelo (best_int8, yolo11s_trained, ...) para poder comparar FP32 vs INT8 sin pisar resultados.
CALIBRATION_OUT_DIR = LOCAL_WORK_DIR / 'calibration' / CALIBRATION_MODEL.stem
argv = [
    '--model', str(CALIBRATION_MODEL),
    '--images', str(PROCESSED_DIR / 'images' / 'test'),
    '--labels', str(PROCESSED_DIR / 'labels' / 'test'),
    '--out-dir', str(CALIBRATION_OUT_DIR),
    '--jpeg-reencode' if JPEG_REENCODE else '--no-jpeg-reencode',
]
if LIMIT:
    argv += ['--limit', str(LIMIT)]
if REFERENCE_METRICS:
    argv += ['--reference-metrics', str(REFERENCE_METRICS)]

calibration = calibrate_thresholds.main(argv)

# Guardar (3 archivos chicos) en Drive y mostrar las curvas.
drive_out = DRIVE_PERSIST_DIR / 'calibration' / CALIBRATION_MODEL.stem
shutil.copytree(CALIBRATION_OUT_DIR, drive_out, dirs_exist_ok=True)
print('Copiado a Drive:', drive_out)

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(CALIBRATION_OUT_DIR / 'threshold_calibration.png')))

### 8d. Diagnosticar las falsas alarmas de una clase

Para una clase débil (por defecto `blob_over_extrusion`), al umbral desplegado (0.25) esta celda clasifica **cada falsa
alarma y cada caja real que el modelo no encontró** según su causa, y guarda imágenes anotadas para mirarlas. Sirve
para decidir el remedio antes de reentrenar. Usa el mismo modelo, pre/postprocesado, reglas de etiquetas y matching que
8c, así que los TP/FP/FN coinciden con la tabla de 8c a ese umbral.

Causas de una **falsa alarma**: `duplicate` (repite un blob real ya detectado), `mislocalized` (sobre un blob real pero
mal encuadrado), `confused_with:X` (sobre un defecto real de otra clase X: el modelo ve un defecto, se equivoca de tipo),
`unannotated` (no hay etiqueta ahí: fondo **o un defecto real que nadie etiquetó** - solo mirando las imágenes se
distingue). Causas de un **blob no encontrado**: `below_threshold`, `predicted_as:X`, `mislocalized`, `not_detected`.

Cómo leerlo: mucho `confused_with` -> problema de clasificación (más datos/mejor separación entre clases); mucho
`unannotated` que en las imágenes parece blob real -> huecos en las etiquetas del dataset, no del modelo; mucho
`unannotated` que es fondo -> faltan negativos; si los blobs perdidos son mucho más chicos que los encontrados (el script
imprime las medianas en píxeles del input de 640) -> objetos pequeños: probar mayor resolución.

Tarda unos minutos (una inferencia por imagen). Deja en Drive `persisted/diagnosis/<clase>/` con las imágenes, los
montajes y `diagnosis.json`.

In [ ]:
!pip install -q onnxruntime

import shutil
import sys

# Modelo desplegado (FP32): el exportado en la celda 8, o el que dejaste en persisted/.
DIAG_MODEL = Path(onnx_fp32_path) if 'onnx_fp32_path' in globals() else DRIVE_PERSIST_DIR / 'best_fp32.onnx'
assert DIAG_MODEL.is_file(), f'No encontré {DIAG_MODEL}. Corré la celda 8 o copiá best_fp32.onnx a {DRIVE_PERSIST_DIR}.'
DIAG_CLASS = 'blob_over_extrusion'   # cualquiera de las 6 clases
DIAG_THRESHOLD = 0.25                # el umbral desplegado en el worker
DIAG_LIMIT = None                    # p. ej. 50 para probar rápido; None = las 841 imágenes

sys.path.insert(0, str(SCRIPTS_DIR))
import inspect_false_positives

DIAG_OUT_DIR = LOCAL_WORK_DIR / 'diagnosis'
argv = [
    '--model', str(DIAG_MODEL),
    '--class-name', DIAG_CLASS,
    '--threshold', str(DIAG_THRESHOLD),
    '--images', str(PROCESSED_DIR / 'images' / 'test'),
    '--labels', str(PROCESSED_DIR / 'labels' / 'test'),
    '--out-dir', str(DIAG_OUT_DIR),
]
if DIAG_LIMIT:
    argv += ['--limit', str(DIAG_LIMIT)]
diagnosis = inspect_false_positives.main(argv)

drive_diag = DRIVE_PERSIST_DIR / 'diagnosis' / DIAG_CLASS
shutil.copytree(DIAG_OUT_DIR / DIAG_CLASS, drive_diag, dirs_exist_ok=True)
print('Copiado a Drive:', drive_diag)

from IPython.display import Image as IPImage, display

for group, title in [('false_positives', 'FALSAS ALARMAS (rojo = falsa alarma, amarillo = acierto, verde = blob real, violeta = blob perdido, azul = otra clase real)'),
                     ('missed', 'BLOBS REALES PERDIDOS')]:
    sheet = DIAG_OUT_DIR / DIAG_CLASS / f'montage_{group}.jpg'
    if sheet.is_file():
        print(f'\n{title}')
        display(IPImage(filename=str(sheet)))

## 9. Bajar los .onnx a tu máquina

Los dos modelos ya están en Drive (`persisted/best_fp32.onnx` y `persisted/best_int8.onnx`, persisten entre
sesiones). Esta sección verifica que sean archivos distintos y dispara la descarga de ambos al navegador.

Copialos a `inference/models/` **con su propio nombre** y apuntá el worker con `onnx_filename` en
`inference/config.py` (ver la sección 4 del README) - no pises `yolo11s_dummy.onnx`.

In [ ]:
import hashlib

md5 = {p.name: hashlib.md5(p.read_bytes()).hexdigest() for p in (DRIVE_PERSIST_DIR / 'best_fp32.onnx', DRIVE_PERSIST_DIR / 'best_int8.onnx')}
for name, digest in md5.items():
    print(f'{name}: {(DRIVE_PERSIST_DIR / name).stat().st_size / 1e6:.1f} MB  md5={digest}')
assert len(set(md5.values())) == 2, 'FP32 e INT8 son el MISMO archivo - algo salió mal en el export.'

In [ ]:
from google.colab import files

files.download(str(DRIVE_PERSIST_DIR / 'best_fp32.onnx'))
files.download(str(DRIVE_PERSIST_DIR / 'best_int8.onnx'))